# Plume sur Google Colab : entraîner le détecteur et ouvrir l'application

Ce carnet fait tout : il récupère le code, fabrique les données d'entraînement, entraîne le modèle sur le
GPU gratuit de Colab, l'évalue, puis ouvre l'application Plume avec le vrai modèle.

**À faire une seule fois :**

1. Créez une clé sur [openrouter.ai/keys](https://openrouter.ai/keys) et ajoutez environ 15 $ de crédit
   ([openrouter.ai/credits](https://openrouter.ai/credits)). Le premier entraînement (200 documents) coûte
   environ 8 à 12 $ ; le carnet ne dépasse jamais `BUDGET_USD`.
2. Menu **Exécution → Modifier le type d'exécution → GPU T4**, puis Enregistrer.
3. Icône 🔑 (Secrets) dans la barre de gauche → **Ajouter un secret** nommé `OPENROUTER_API_KEY`, collez la
   clé, activez « Accès au notebook ». Sans secret, le carnet vous demandera la clé.
4. Menu **Exécution → Tout exécuter**, puis autorisez l'accès à Google Drive.

Tout est enregistré dans **Mon Drive/Plume** (données, modèle, rapports). Si Colab se déconnecte, relancez
« Tout exécuter » : il reprend où il s'était arrêté, sans repayer ce qui est déjà généré.
Durée : environ 30 à 45 minutes pour 200 documents.

**Déjà entraîné ?** « Tout exécuter » saute les étapes faites et ouvre directement l'application (le GPU
n'est alors plus nécessaire).

**Les meilleures données :** des IA, EE ou essais TOK écrits **avant 2023** (.docx, .pdf, .txt), déposés dans
**Mon Drive/Plume/data/human/local** avant de lancer. Ils ne quittent jamais votre Drive.

In [ ]:
#@title Réglages
#@markdown Nombre de documents humains utilisés. 200 ≈ 8–12 $ ; la version complète : 1500 documents et un budget de 60 $.
MAX_DOCS = 200  #@param {type:"integer"}
#@markdown Plafond total des dépenses OpenRouter (en $) pour ce carnet, tous tours compris.
BUDGET_USD = 15  #@param {type:"number"}
#@markdown 2e tour : cherche les textes humains que le modèle accuse à tort et s'entraîne dessus (moins de faux positifs).
ROUND_2 = True  #@param {type:"boolean"}
#@markdown Cochez pour ré-entraîner un modèle déjà fait (par exemple après avoir augmenté MAX_DOCS).
RETRAIN = False  #@param {type:"boolean"}
BASE_MODEL = "FacebookAI/xlm-roberta-base"  #@param ["FacebookAI/xlm-roberta-base", "FacebookAI/xlm-roberta-large", "intfloat/multilingual-e5-small"]
EPOCHS = 3  #@param {type:"number"}
#@markdown Sans Drive, tout est perdu à la déconnexion : laissez coché.
USE_DRIVE = True  #@param {type:"boolean"}
BRANCH = "claude/lucid-clarke-fcjgco"  #@param {type:"string"}

REPO_URL = "https://github.com/Deonandre/For-la-croqueta-.git"
REPO_DIR = "/content/For-la-croqueta-"
TMP_DIR = "/content/plume-tmp"  # fast local disk for training checkpoints
ALLOW_CPU_TRAINING = False      # hours on CPU; only sensible with intfloat/multilingual-e5-small

In [ ]:
#@title 1. Installation
import codecs, copy, json, os, shutil, subprocess, sys, time
from pathlib import Path


def sh(*cmd, check=True):
    """Run a command, streaming its output (progress bars included) into the notebook."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    while chunk := p.stdout.read1(65536):
        sys.stdout.write(dec.decode(chunk))
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"La commande a échoué (code {p.returncode}) : {' '.join(cmd)}")
    return p.returncode


def plume(*args, check=True):
    return sh(sys.executable, "-m", "aidetect", *args, check=check)


if Path(REPO_DIR, ".git").exists():  # re-run: update to the latest code, data lives outside the clone
    sh("git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", BRANCH)
    sh("git", "-C", REPO_DIR, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD")
else:
    sh("git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR)
WORK = Path(REPO_DIR, "ai-detector")
os.chdir(WORK)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

import torch
GPU = torch.cuda.is_available()
print("GPU :", torch.cuda.get_device_name(0) if GPU else "aucun (Exécution → Modifier le type d'exécution → GPU T4)")

In [ ]:
#@title 2. Stockage dans Google Drive (Mon Drive/Plume)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/Plume")
else:
    STORE = Path("/content/plume-store")

# The clone only holds code; data, models, reports and the paid LLM cache live in STORE.
for name, rel in [("data", "data"), ("models", "models"), ("reports", "reports"), ("llm_cache", ".cache/llm")]:
    target, link = STORE / name, WORK / rel
    target.mkdir(parents=True, exist_ok=True)
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f"{link} existe déjà : supprimez-le puis relancez.")
    link.symlink_to(target, target_is_directory=True)
LOCAL = STORE / "data" / "human" / "local"
LOCAL.mkdir(parents=True, exist_ok=True)
print("Stockage :", STORE)
print("Essais d'élèves (avant 2023) :", sum(1 for f in LOCAL.rglob("*") if f.suffix.lower() in {".txt", ".md", ".docx", ".pdf"}),
      "fichier(s) dans", LOCAL)

import yaml

BASE_CFG = yaml.safe_load(Path("configs/default.yaml").read_text())
DATA = Path("data/examples.jsonl")
HARD = Path("data/hard_negatives.jsonl")
MODEL_R1 = Path("models/plume-r1")   # first round, used to mine hard negatives
MODEL = Path("models/plume")         # final model


def spent_usd():
    """Total OpenRouter spend of this notebook so far, from the cache every paid call leaves behind."""
    return sum(json.loads(p.read_text()).get("cost", 0) for p in Path(".cache/llm").glob("*.json"))


def write_cfg(budget):
    cfg = copy.deepcopy(BASE_CFG)
    cfg["generation"]["budget_usd"] = round(max(0.0, budget), 2)
    cfg["training"]["base_model"] = BASE_MODEL
    cfg["training"]["epochs"] = EPOCHS
    # mining scans a pool a few times larger than what the build used (documents already used are skipped)
    cfg["mining_pool"] = [dict(s, limit=min(s["limit"] * 3, MAX_DOCS * 2)) if "limit" in s else s
                          for s in cfg["human_sources"]]
    Path("configs/colab.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
    return "configs/colab.yaml"


def trained(path):
    return (path / "calibration.json").exists()  # written last by `train`


def train_to(dest):
    """Train on the fast local disk, then keep only the final model (no checkpoints) in STORE."""
    if not GPU and not ALLOW_CPU_TRAINING:
        raise SystemExit("Pas de GPU. Menu Exécution → Modifier le type d'exécution → GPU T4, puis « Tout exécuter ».")
    tmp = Path(TMP_DIR) / dest.name
    shutil.rmtree(tmp, ignore_errors=True)
    plume("train", "--config", write_cfg(0), "--out", tmp)
    publish(tmp, dest)
    shutil.rmtree(tmp, ignore_errors=True)


def publish(src, dest):
    """Copy a model folder into place in one step, so a disconnect never leaves half a model."""
    staging = dest.with_name(dest.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    shutil.copytree(src, staging, ignore=shutil.ignore_patterns("checkpoints"))
    shutil.rmtree(dest, ignore_errors=True)
    staging.rename(dest)


# Only a finished model is wiped: re-running with RETRAIN still ticked after a disconnect resumes instead.
if RETRAIN and trained(MODEL):
    for p in (MODEL, MODEL_R1):
        shutil.rmtree(p, ignore_errors=True)
    HARD.unlink(missing_ok=True)
DONE = trained(MODEL)
if DONE:
    print("Modèle déjà entraîné : les étapes 3 à 7 sont sautées (cochez RETRAIN pour recommencer).")

In [ ]:
#@title 3. Clé OpenRouter et vérification
def get_key():
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"]
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key
    except Exception:  # no secret, or notebook access not granted
        pass
    from getpass import getpass
    return getpass("Collez votre clé OpenRouter (sk-or-...) puis Entrée : ").strip()


if not DONE:
    if not GPU and not ALLOW_CPU_TRAINING:  # stop before paying for data that could not be trained on
        raise SystemExit("Pas de GPU. Menu Exécution → Modifier le type d'exécution → GPU T4, puis « Tout exécuter ».")
    os.environ["OPENROUTER_API_KEY"] = get_key()
    plume("doctor", "--config", write_cfg(BUDGET_USD), "--require", "build")
    plume("models", "--config", write_cfg(BUDGET_USD), check=False)

In [ ]:
#@title 4. Données d'entraînement (≈ 10–20 min pour 200 documents)
from collections import Counter
from html import escape
from IPython.display import HTML, Markdown, display


def show_data():
    rows = [json.loads(l) for l in DATA.open(encoding="utf-8") if l.strip()]
    if not rows:
        raise SystemExit("Aucun exemple généré : vérifiez la clé et le crédit OpenRouter (messages ci-dessus).")
    docs = {r["doc_id"] for r in rows}
    print(f"{len(rows)} exemples tirés de {len(docs)} documents humains ; dépensé : {spent_usd():.2f} $")
    for key in ("split", "variant", "lang", "source"):
        print(f"  {key:8}", dict(Counter(r[key] for r in rows).most_common()))
    # one mixed document, coloured like the app: check by eye that the labels make sense
    ex = next((r for r in rows if r["variant"] == "mosaic"), rows[0])
    colours = {1: "#fecaca", 2: "#fde68a"}
    parts = []
    for s, lab, para in zip(ex["sentences"], ex["labels"], ex["para_starts"]):
        parts.append(("<br><br>" if para and parts else "") +
                     f'<span style="background:{colours.get(lab, "none")};color:#111">{escape(s)}</span> ')
    display(HTML(f"<p><b>Exemple « {ex['variant']} » ({ex['lang']})</b> : rouge = IA, jaune = reformulé par IA</p>"
                 f"<div style='max-width:860px;line-height:1.6;background:#fff;padding:12px;border-radius:8px'>{''.join(parts)}</div>"))


if not DONE:
    budget = BUDGET_USD - spent_usd()
    print(f"Budget restant : {budget:.2f} $")
    plume("build", "--config", write_cfg(budget), "--max-docs", MAX_DOCS)
if DATA.exists():
    show_data()

In [ ]:
#@title 5. Référence rapide (sans GPU) : le score à battre
if not DONE:
    plume("baseline", "--out", "models/baseline.joblib")
    plume("eval", "--model", "models/baseline.joblib", "--out", "reports/baseline")
    display(Markdown(Path("reports/baseline/report.md").read_text()))

In [ ]:
#@title 6. Entraînement du modèle, tour 1 (GPU, ≈ 10–20 min)
if not DONE and not trained(MODEL_R1):
    train_to(MODEL_R1)

In [ ]:
#@title 7. Tour 2 : textes humains accusés à tort → nouvel entraînement
if not DONE:
    budget = BUDGET_USD - spent_usd()
    n_hard = 0
    if ROUND_2 and budget >= 1:
        if not HARD.exists():
            plume("mine", "--model", MODEL_R1, "--config", write_cfg(budget), "--top-k", max(50, MAX_DOCS // 2))
        n_hard = sum(1 for line in HARD.open(encoding="utf-8") if line.strip())
    if n_hard:
        plume("build", "--config", write_cfg(budget), "--human-jsonl", HARD)
        show_data()
        train_to(MODEL)
    else:
        if ROUND_2 and budget < 1:
            print(f"Budget restant trop faible ({budget:.2f} $) : tour 2 sauté.")
        elif ROUND_2:
            print("Aucun texte humain accusé à tort : pas besoin de tour 2.")
        print("Le modèle du tour 1 devient le modèle final.")
        publish(MODEL_R1, MODEL)
    DONE = trained(MODEL)

In [ ]:
#@title 8. Évaluation sur des textes jamais vus (dont 2 familles de modèles jamais vues)
REPORT = Path("reports/plume/report.md")
if not REPORT.exists() or REPORT.stat().st_mtime < (MODEL / "calibration.json").stat().st_mtime:
    plume("eval", "--model", MODEL, "--out", "reports/plume")
print("Le chiffre clé est le taux de faux positifs sur les phrases humaines (human FPR) : il doit rester bas.")
display(Markdown(REPORT.read_text()))
if RETRAIN:
    print("Ré-entraînement terminé : décochez RETRAIN, sinon le prochain « Tout exécuter » recommencera.")
print(f"Dépense totale OpenRouter : {spent_usd():.2f} $ (plafond {BUDGET_USD} $)")

In [ ]:
#@title 9. Ouvrir Plume
import httpx

if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait()
server = subprocess.Popen([sys.executable, "-m", "aidetect", "serve", "--model", str(MODEL), "--port", "8000"],
                          stdout=open("/tmp/plume-server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):  # the first health check loads the model
    try:
        # one request at a time: it blocks while the model loads (a timeout would start another load)
        if httpx.get("http://127.0.0.1:8000/api/health", timeout=300).json().get("ready"):
            break
    except httpx.HTTPError:
        pass
    if server.poll() is not None:
        raise RuntimeError("Le serveur s'est arrêté :\n" + Path("/tmp/plume-server.log").read_text()[-3000:])
    time.sleep(1)
else:
    raise RuntimeError("Le serveur ne répond pas :\n" + Path("/tmp/plume-server.log").read_text()[-3000:])

from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8000)")
display(HTML(f'<p style="font-size:18px">✅ Plume est prêt : <a href="{url}" target="_blank">ouvrir l\'application</a></p>'
             "<p>Le lien marche tant que ce carnet reste ouvert. Les textes analysés ne sont jamais enregistrés.</p>"))

## Et ensuite ?

- **Version complète** : `MAX_DOCS = 1500`, `BUDGET_USD = 60`, cochez `RETRAIN`, puis « Tout exécuter ».
  Les documents déjà générés ne sont pas repayés.
- **Moins de faux positifs** : ajoutez des essais d'avant 2023 dans `Mon Drive/Plume/data/human/local`, puis
  relancez avec `RETRAIN` coché.
- **Avis de Claude** : copiez le rapport de l'étape 8 (`Mon Drive/Plume/reports/plume/report.md`) dans la
  conversation.
- **Utiliser Plume plus tard** : rouvrez ce carnet et « Tout exécuter » : il saute l'entraînement et ouvre
  l'application en une minute.